# 02 · Your first Iceberg table with PySpark

**Time:** ~25 min · **You will learn:** create a table, insert rows with SQL and with DataFrames, query and aggregate it.

In [ ]:
from helpers import get_spark, ORDERS_DDL
from datetime import datetime

spark = get_spark("02-first-table")

## 1. Create a table
`USING iceberg` is what makes it an Iceberg table. We start from a clean slate so the notebook can be re-run.

In [ ]:
spark.sql("DROP TABLE IF EXISTS demo.course.orders PURGE")

spark.sql("""
    CREATE TABLE demo.course.orders (
        order_id   BIGINT,
        customer   STRING,
        product    STRING,
        quantity   INT,
        unit_price DOUBLE,
        order_ts   TIMESTAMP
    ) USING iceberg
""")
spark.sql("DESCRIBE demo.course.orders").show(truncate=False)

## 2. Insert with SQL

In [ ]:
spark.sql("""
    INSERT INTO demo.course.orders VALUES
        (1001, 'alice', 'keyboard', 1, 49.0,  TIMESTAMP '2024-01-01 09:00:00'),
        (1002, 'bob',   'mouse',    2, 19.5,  TIMESTAMP '2024-01-01 16:00:00')
""")
spark.table("demo.course.orders").show()

## 3. Insert with the DataFrame API
`writeTo(...).append()` is the DataFrame equivalent of `INSERT INTO`. Each call is one atomic **commit**.

In [ ]:
new_rows = [
    (1003, "carol", "monitor", 1, 189.0, datetime(2024, 1, 2, 11, 30)),
    (1004, "alice", "laptop",  1, 999.0, datetime(2024, 1, 2, 15, 45)),
]
df = spark.createDataFrame(new_rows, schema=ORDERS_DDL)
df.writeTo("demo.course.orders").append()

spark.table("demo.course.orders").orderBy("order_id").show()

## 4. Query it like any Spark table

In [ ]:
from pyspark.sql import functions as F

orders = spark.table("demo.course.orders")

(orders
    .withColumn("revenue", F.col("quantity") * F.col("unit_price"))
    .groupBy("customer")
    .agg(F.sum("revenue").alias("total_revenue"), F.count("*").alias("orders"))
    .orderBy(F.desc("total_revenue"))
    .show())

The same thing in SQL:

In [ ]:
spark.sql("""
    SELECT customer, SUM(quantity * unit_price) AS total_revenue, COUNT(*) AS orders
    FROM demo.course.orders
    GROUP BY customer
    ORDER BY total_revenue DESC
""").show()

## 5. Inspect the table definition

In [ ]:
spark.sql("SHOW CREATE TABLE demo.course.orders").show(truncate=False)

Notice the `TBLPROPERTIES` (format version, write modes) and the `LOCATION` in `s3://warehouse/...`:
your data really lives in object storage, not in the container.

### Your turn

Append three more orders with the DataFrame API (ids 1005 to 1007, any values), then compute the
**average order value** (`quantity * unit_price`) per product. Assert that the table now has 7 rows.

In [ ]:
# Write your solution here


<details>
<summary>Show solution</summary>

```python
more = [
    (1005, "bob",   "keyboard", 2, 49.0,  datetime(2024, 1, 3, 10, 0)),
    (1006, "dave",  "mouse",    3, 19.5,  datetime(2024, 1, 3, 12, 0)),
    (1007, "carol", "laptop",   1, 999.0, datetime(2024, 1, 3, 18, 0)),
]
spark.createDataFrame(more, schema=ORDERS_DDL).writeTo("demo.course.orders").append()
assert spark.table("demo.course.orders").count() == 7

(spark.table("demo.course.orders")
    .groupBy("product")
    .agg(F.avg(F.col("quantity") * F.col("unit_price")).alias("avg_order_value"))
    .show())
```

</details>

### Recap
* `CREATE TABLE ... USING iceberg` creates a table; `INSERT INTO` and `writeTo().append()` add data.
* Every write is an atomic commit. Reads always see a consistent snapshot.
* Once registered, an Iceberg table behaves like any other Spark table.

Next: **03 · Under the hood**: what did those commits actually write?